# Notebook 02 — Baseline Models
### Paper: Profit-Driven Telecom Churn Prediction (IEEE CSDE 2026)

**Input:** cleaned splits from Notebook 01 (add that notebook's output as input, or re-run Notebook 01 above).

Expected files in `INPUT_DIR`:
- `maven_train.csv` / `maven_cal.csv` / `maven_test.csv` (+ `*_feature_meta.csv`)
- `cell_train.csv` / `cell_cal.csv` / `cell_test.csv`

**What it does:**
- **Leakage-safe preprocessing** (impute/scale/one-hot fit on TRAIN only)
  - numeric: median impute + missing-indicator (handles `HandsetPrice` ~57% NaN)
  - categorical: constant-fill + one-hot (unknown-safe)
  - scaling applied for Logistic Regression only
- **Baselines at natural class distribution** (no resampling) — this is the reference point; imbalance ablation + calibration come in later notebooks
- **Full metric suite:** discrimination + calibration baseline (Brier, ECE, LogLoss)
- **Saves** fitted pipelines + predicted probabilities on CAL and TEST splits (cal-split probs are needed later for calibration + profit thresholding)

In [1]:
# ===== CELL 1: imports + config + load =====
import os, json, warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")
 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             recall_score, precision_score, brier_score_loss,
                             log_loss, accuracy_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
 
SEED = 42
np.random.seed(SEED)
 
# If using Notebook 01 output directly in the same session:
INPUT_DIR = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1"

 
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
 
DATASETS = {
    "maven": ("maven_train.csv", "maven_cal.csv", "maven_test.csv"),
    "cell":  ("cell_train.csv",  "cell_cal.csv",  "cell_test.csv"),
}
 

In [2]:
# ===== CELL 2: helpers — preprocessor + ECE =====
def load_split(name):
    tr, ca, te = DATASETS[name]
    train = pd.read_csv(f"{INPUT_DIR}/{tr}")
    cal   = pd.read_csv(f"{INPUT_DIR}/{ca}")
    test  = pd.read_csv(f"{INPUT_DIR}/{te}")
    return train, cal, test
 
def split_xy(df):
    y = df["target"].values
    X = df.drop(columns=["target"])
    return X, y
 
def build_preprocessor(X, scale=False):
    """Leakage-safe: fit on train only. Numeric median-impute + missing-indicator,
       categorical constant-fill + one-hot. Optional scaling (for LR)."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
 
    num_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)
 
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
 
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
 
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """Standard ECE with equal-width bins."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins) - 1
    idx = np.clip(idx, 0, n_bins - 1)
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece
 
def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUC":       roc_auc_score(y_true, y_prob),
        "PR_AUC":    average_precision_score(y_true, y_prob),
        "F1":        f1_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Brier":     brier_score_loss(y_true, y_prob),
        "LogLoss":   log_loss(y_true, y_prob, labels=[0, 1]),
        "ECE":       expected_calibration_error(y_true, y_prob),
    }
 

In [3]:
# ===== CELL 3: model factory =====
def get_models():
    """Natural-distribution baselines (no class weighting here — that's the ablation)."""
    return {
        "LogReg": (LogisticRegression(max_iter=2000, random_state=SEED), True),   # needs scaling
        "RF":     (RandomForestClassifier(n_estimators=400, n_jobs=-1,
                                          random_state=SEED), False),
        "XGB":    (XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.9,
                                 tree_method="hist", eval_metric="logloss",
                                 random_state=SEED, n_jobs=-1), False),
        "LGBM":   (LGBMClassifier(n_estimators=400, max_depth=-1, learning_rate=0.05,
                                  subsample=0.9, colsample_bytree=0.9,
                                  random_state=SEED, n_jobs=-1, verbose=-1), False),
    }

In [4]:
# ===== CELL 4: train + evaluate all models on both datasets =====
rows = []
prob_store = {}   # (dataset, model) -> {"cal": probs, "test": probs}
 
for ds in DATASETS:
    train, cal, test = load_split(ds)
    X_tr, y_tr = split_xy(train)
    X_ca, y_ca = split_xy(cal)
    X_te, y_te = split_xy(test)
    print(f"\n===== {ds.upper()} =====  train={X_tr.shape} cal={X_ca.shape} test={X_te.shape}")
 
    for mname, (clf, needs_scale) in get_models().items():
        pre = build_preprocessor(X_tr, scale=needs_scale)
        pipe = Pipeline([("pre", pre), ("clf", clf)])
        pipe.fit(X_tr, y_tr)
 
        p_ca = pipe.predict_proba(X_ca)[:, 1]
        p_te = pipe.predict_proba(X_te)[:, 1]
 
        m_te = evaluate(y_te, p_te)
        m_te.update({"dataset": ds, "model": mname, "split": "test"})
        rows.append(m_te)
 
        # save fitted pipeline + probs for later notebooks
        joblib.dump(pipe, f"{OUT_DIR}/model_{ds}_{mname}.joblib")
        prob_store[(ds, mname)] = {"cal": p_ca, "test": p_te}
        np.save(f"{OUT_DIR}/prob_{ds}_{mname}_cal.npy",  p_ca)
        np.save(f"{OUT_DIR}/prob_{ds}_{mname}_test.npy", p_te)
 
        print(f"  {mname:6s} | AUC={m_te['AUC']:.4f} PR={m_te['PR_AUC']:.4f} "
              f"F1={m_te['F1']:.4f} Brier={m_te['Brier']:.4f} ECE={m_te['ECE']:.4f}")
 
# also save y_true for cal/test per dataset (needed later)
for ds in DATASETS:
    _, cal, test = load_split(ds)
    np.save(f"{OUT_DIR}/y_{ds}_cal.npy",  cal["target"].values)
    np.save(f"{OUT_DIR}/y_{ds}_test.npy", test["target"].values)


===== MAVEN =====  train=(3953, 30) cal=(1318, 30) test=(1318, 30)
  LogReg | AUC=0.9043 PR=0.7961 F1=0.7075 Brier=0.1097 ECE=0.0309
  RF     | AUC=0.9190 PR=0.8471 F1=0.7313 Brier=0.0988 ECE=0.0232
  XGB    | AUC=0.9251 PR=0.8621 F1=0.7596 Brier=0.0955 ECE=0.0346
  LGBM   | AUC=0.9252 PR=0.8600 F1=0.7566 Brier=0.0986 ECE=0.0575

===== CELL =====  train=(30628, 55) cal=(10209, 55) test=(10210, 55)
  LogReg | AUC=0.6248 PR=0.3940 F1=0.0783 Brier=0.1970 ECE=0.0098
  RF     | AUC=0.6742 PR=0.4491 F1=0.1215 Brier=0.1895 ECE=0.0239
  XGB    | AUC=0.6846 PR=0.4648 F1=0.2351 Brier=0.1868 ECE=0.0083
  LGBM   | AUC=0.6788 PR=0.4652 F1=0.2477 Brier=0.1875 ECE=0.0102


In [5]:
# ===== CELL 5: results table =====
results = pd.DataFrame(rows)[
    ["dataset", "model", "AUC", "PR_AUC", "F1", "Recall", "Precision",
     "Accuracy", "Brier", "LogLoss", "ECE"]
].round(4)
results = results.sort_values(["dataset", "AUC"], ascending=[True, False]).reset_index(drop=True)
results.to_csv(f"{OUT_DIR}/results_baseline.csv", index=False)
print("\n================ BASELINE RESULTS (test) ================")
print(results.to_string(index=False))


================ BASELINE RESULTS (test) ================
dataset  model    AUC  PR_AUC     F1  Recall  Precision  Accuracy  Brier  LogLoss    ECE
   cell    XGB 0.6846  0.4648 0.2351  0.1482     0.5684    0.7221 0.1868   0.5552 0.0083
   cell   LGBM 0.6788  0.4652 0.2477  0.1570     0.5863    0.7252 0.1875   0.5570 0.0102
   cell     RF 0.6742  0.4491 0.1215  0.0673     0.6266    0.7197 0.1895   0.5617 0.0239
   cell LogReg 0.6248  0.3940 0.0783  0.0421     0.5463    0.7139 0.1970   0.5807 0.0098
  maven   LGBM 0.9252  0.8600 0.7566  0.7273     0.7884    0.8672 0.0986   0.3467 0.0575
  maven    XGB 0.9251  0.8621 0.7596  0.7139     0.8116    0.8718 0.0955   0.3085 0.0346
  maven     RF 0.9190  0.8471 0.7313  0.6658     0.8111    0.8612 0.0988   0.3148 0.0232
  maven LogReg 0.9043  0.7961 0.7075  0.6952     0.7202    0.8369 0.1097   0.3490 0.0309


In [6]:
# ===== CELL 6 (FIXED, optional): 5-fold CV AUC — Maven only, no nested parallelism =====
from sklearn.model_selection import cross_val_score, StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for ds in ["maven"]:                      # Cell2Cell CV skip — slow, low-signal
    train, _, _ = load_split(ds)
    X_tr, y_tr = split_xy(train)
    for mname, (clf, needs_scale) in get_models().items():
        pipe = Pipeline([("pre", build_preprocessor(X_tr, scale=needs_scale)), ("clf", clf)])
        auc = cross_val_score(pipe, X_tr, y_tr, cv=skf, scoring="roc_auc", n_jobs=1)  # outer=1, model uses all cores
        cv_rows.append({"dataset": ds, "model": mname,
                        "CV_AUC_mean": auc.mean(), "CV_AUC_std": auc.std()})
        print(f"{ds} {mname:6s} CV-AUC = {auc.mean():.4f} ± {auc.std():.4f}")
pd.DataFrame(cv_rows).round(4).to_csv(f"{OUT_DIR}/results_cv_auc.csv", index=False)

maven LogReg CV-AUC = 0.9099 ± 0.0138
maven RF     CV-AUC = 0.9221 ± 0.0141
maven XGB    CV-AUC = 0.9297 ± 0.0110
maven LGBM   CV-AUC = 0.9284 ± 0.0101
